# Chat templates

You have your dataset, but how do you feed it to the language model? 

We need to convert the dataset to a format that the model can use for the finetuning task.  Enter 'chat templates'. 

To adapt a model for conversational tasks, chat templates define the input and output structure. They establish a consistent conversational framework, addressing how distinct roles (e.g. user, system, assistant) interact and how their messages are represented as tokens.

### Key Components:

- **Role Interaction:** - templates distinguish between the following roles in a conversation:
    - System: Provides instructions or context for the assistant (some chat templates do not use the system role).
    - User: Asks questions or provides input.
    - Assistant: Responds based on instructions and context.

- **Message Formatting:** 
  - Ensures that the structure of the conversation (e.g., who said what) is encoded into the input sequence.
  - May include tokens or delimiters to demarcate different roles and maintain clarity.

### Why Templates Matter:

- Different language models have been trained on specific data formats during an intial instruction tuning. ⚠️ Mismatch between the chat template used for inference and the format the model was trained on can lead to degraded performance. 

- Templates ensure consistency and allow the model to fully leverage its fine-tuned capabilities without confusion or errors.



### Example template 

```text
<|system|>
{{ INSTRUCTIONS FOR THE MODEL GOES HERE}} 
<|end|>

<|user|>
{{ USER'S QUERY GOES HERE }}

<|assistant|>
{{ MODEL'S OUTPUT GOES HERE }}
<|endoftext|>
```

### Inspecting the chat template used by a tokenizer 

You can view the chat template used by a particular tokenizer using `tokenizer.chat_template`. 

For example, this is the chat template for Phi-3.5-mini-instruct's tokenizer: 

![](images/tokenizer-chat-template.png)

The tokenizer also has a '.apply_chat_template()' function which we will use below... 

----

## Providing guidance to the model 

Reminder: The model's job is to take in something like this:

`USER QUERY: Set the temperature to 50 degrees`

And output something like this:

`set_temperature(temperature=50)`


However, empirically, it is better to provide some instructions to the model to make the learning task easier for the model, so ideally, we should have something like this:

```
INSTRUCTIONS: You need to call a function to satisfy the user's request!

USER QUERY: Set the temperature to 50 degrees
```

And output:

`set_temperature(temperature=50)`


<span style="color: green;">**QUESTION**</span>: Why is this a good format?  Why can't we say "USER'S SPEECH" instead of "USER'S QUERY", or where did "USER QUERY" came from?

<details>
    <summary>Answer: Click to show</summary>

That depends on the tokenizer that the model was trained with. If the model was trained to work with "USER QUERY" then it is better to use "USER QUERY" in the chat template. To make sure you use the correct chat template, you can use the chat template provided by the tokenizer that was packaged with the model.
</details>


🤔  **TRY FOR YOURSELF: Write down the instructions you would use for the SLM before proceeding ...** 


---

## Creating a function to format data with the tokenizer's chat template 

We have created the following function to format a sample from the dataset with a chat template and add it to the data sample. 

In [ ]:
from textwrap import dedent

from transformers import PreTrainedTokenizerBase
from typing import Any

# Define a system prompt for an in-car virtual assistant.
# This prompt guides the behavior of the assistant by specifying how it should respond to user inputs.
SYSTEM_PROMPT = dedent(
    """
    You are an in car virtual assistant that maps user's inputs to the corresponding function call in the vehicle. You must respond with only a JSON object matching the following schema:

    {
        "function_name": <name of the function>,
        "parameters": {
            <arg1_name>: <arg1_value>,
            <arg2_name>: <arg2_value>,
            ...
        }
    }

    Important: DO NOT include any other information in the response.
    """
)


def apply_chat_template(
        example: dict[str, Any],
        tokenizer: PreTrainedTokenizerBase,
    ) -> dict[str, Any]:
    """
    Prepares an input example for a chat-based model by applying a chat template.

    Args:
    -----
    example (dict[str, Any]): A dictionary containing the input example. It must include keys like:
        - "prompt": The user's input or question.
        - "ground_truth": The expected response or reference answer from the assistant.
    tokenizer (PreTrainedTokenizerBase): A tokenizer object from the Hugging Face Transformers library.
        It is used to format and prepare the input for the model.

    Returns:
    --------
    dict[str, Any]: The updated example dictionary with a new "text" key that contains the formatted
                    input for the chat-based model.
    """
    # Create a message chain that defines the context and dialogue flow for the chat-based model.
    # This includes:
    # 1. A system prompt to instruct the assistant on how to behave or respond.
    # 2. A user message from the example's "prompt" key.
    # 3. An assistant message representing the ground truth response.
    message_chain = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,  # System-level instruction for the assistant's behavior.
        },
        {
            "role": "user",
            "content": example["prompt"],  # User-provided input or question.
        },
        {
            "role": "assistant",
            "content": example["ground_truth"],  # Reference response from the assistant.
        },
    ]

    # Apply the tokenizer's chat template.
    # - `message_chain`: The structured conversation to be tokenized.
    # - `tokenize=False`: Keeps the text in its raw form instead of tokenizing it.
    # - `add_generation_prompt=False`: Prevents adding a generation-specific prompt, leaving the response open-ended.
    example["text"] = tokenizer.apply_chat_template(
        message_chain,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Return the updated example dictionary with the "text" key added.
    return example


## 🤔 Exercise: 

Use the Datasets library you saw in the previous notebook to load the training dataset (/Data/train_data.csv), take a sample & then run it through the `apply_chat_template()` function (above). 

You will also need the tokenizer for your model too (use the same model name to use it from cache & avoid downloading another model!)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

train_dataset = load_dataset("csv", data_files="../Data/clean_train_data.csv", split="all")


Inspect a sample from the training data: 


In [ ]:
sample = train_dataset[0]

sample

We will now take a look at the transformed data. Be mindful of  `<|system|>`, `<|user|>`, and `<|assistant|>`, as they are special "words" used to separate the different parts of the input.

In [ ]:
from pprint import pprint

processed_sample = apply_chat_template(example=sample, tokenizer=tokenizer)

pprint(processed_sample["text"], width=120)

Hopefully you now have a better feel for how data (the prompt & ground truth) will be fed to the model for each sample. 